# Multi-view Pseudo-label Generation — Kaggle

Notebook này chạy **Multi-view Pseudo-label Generation**: dùng Source Teacher
(`hotel-mT5` đã fine-tune ASQP) để tạo 3 góc nhìn (views) cho mỗi review chưa
gán nhãn — Native, Translate-to-English, Back-translation — rồi hợp nhất 3 dự
đoán bằng **self-consistency voting** trước khi đưa vào **Architectural
Disagreement** + **Confidence Fusion** hiện có với Extractive Teacher (XLM-R).

```
Unlabeled Review
       │
       ├──► View 1 Native ──────────────► hotel-mT5 ──┐
       ├──► View 2 Translate→EN ────────► hotel-mT5 ──┼──► Self-consistency vote ──► gen_quads
       └──► View 3 Back-translation ────► hotel-mT5 ──┘           │
                                                                    ▼
                                          gen_quads ──┐
                                                       ├──► Architectural Disagreement ──► Confidence Fusion ──► pseudo_labels
                                          ext_quads ───┘        (XLM-R teacher, không đổi)
```

Code cho stage này nằm ở `teacher/translator.py` (NLLB-200 many-to-many MT)
và `teacher/multiview.py` (`MultiViewGenerativeTeacher`), wire vào `train.py`
qua cờ `--multiview`.

## Trước khi chạy

1. **Settings** → bật **GPU** (T4 x2 hoặc P100) và **Internet**.
2. **Add Input** 2 dataset đã train sẵn từ notebook `dual-teacher-kaggle.ipynb`
   trước đó (notebook đó tạo ra đúng 2 file zip này ở cell cuối):
   - `hotel-mt5-asqp.zip` → Generative Teacher đã fine-tune ASQP.
   - `dual-teacher-output.zip` → chứa `extractive_teacher.pt` (Extractive
     Teacher đã train) trong `checkpoints/dual-teacher/`.
3. **Add Input** dataset chứa `hotel_review_merged.csv` (hoặc
   `hotel_review*_lang.csv`) — **phải có cột `language`** (từ
   `add_language.py`/fastText) để multi-view biết chiều dịch cho từng review;
   thiếu cột này thì mọi review bị coi là tiếng Anh (View 2/3 thành no-op).

## Lưu ý hiệu năng

Dịch qua NLLB-200 cho **hàng triệu** review (xem `language_stats.txt`) sẽ rất
lâu trên 1 phiên Kaggle (giới hạn ~9-12h). Nên **thử với `MAX_SAMPLES` nhỏ**
(vài trăm review) trước để kiểm tra pipeline chạy đúng, rồi mới tăng dần hoặc
chia batch chạy nhiều phiên.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GiB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))


In [ ]:
# Bật GPU và Internet trong Settings trước khi chạy
!git clone https://github.com/hotuyen21pt/MultiLABSA.git

# NLLB-200 dùng chung tokenizer/deps với mT5 (sentencepiece + protobuf) —
# requirements-kaggle.txt đã liệt kê đủ, không cần cài thêm gì cho dịch.
!pip install -q -r MultiLABSA/requirements-kaggle.txt


In [ ]:
import os, fnmatch

# Kaggle mount dataset dưới /kaggle/input bằng symlink tới thư mục phiên bản,
# nên phải os.walk(followlinks=True) để duyệt xuyên qua symlink.

# required_files: neu truyen vao, uu tien thu muc KHOP TEN va CHUA THANG 1
# trong cac file do ngay ben trong no - tranh nham voi mot thu muc CUNG TEN
# nhung chi la container cua cac checkpoint-N/ con. Vi du khi Add Input ca
# "Notebook Output" cua notebook dual-teacher-kaggle truoc do, se co CA
# "checkpoints/hotel-mt5-asqp/" (chi chua checkpoint-500/, checkpoint-1000/,
# KHONG co model.safetensors truc tiep) LAN "hotel-mt5-asqp/" (thu muc final
# that su, co model.safetensors truc tiep) - neu khong loc theo
# required_files, os.walk co the gap thu muc container truoc va tra nham no
# ve (day chinh la loi "no file named model.safetensors ... found in
# directory .../checkpoints/hotel-mt5-asqp" o lan chay truoc).
def find_dir_by_name(names, root="/kaggle/input", required_files=None):
    fallback = None
    for dirpath, _dirs, files in os.walk(root, followlinks=True):
        if os.path.basename(dirpath.rstrip("/\\")) in names:
            if not required_files or any(f in files for f in required_files):
                return dirpath
            if fallback is None:
                fallback = dirpath
    return fallback

def find_file(patterns, root="/kaggle/input"):
    for dirpath, _dirs, files in os.walk(root, followlinks=True):
        for fn in files:
            if any(fnmatch.fnmatch(fn, p) for p in patterns):
                return os.path.join(dirpath, fn)
    return None

_MODEL_WEIGHT_FILES = ["model.safetensors", "pytorch_model.bin"]

UNLABELED_CSV = find_file(["hotel_review_merged.csv", "hotel_review*_lang.csv"])
GEN_MODEL_DIR = find_dir_by_name(
    {"hotel-mt5-asqp", "hotel_mt5_asqp", "hotel-mt5_asqp"}, required_files=_MODEL_WEIGHT_FILES
)
EXTRACTIVE_CKPT = find_file(["extractive_teacher.pt"])

print("UNLABELED_CSV   :", UNLABELED_CSV or "KHONG TIM THAY - can Add Input dataset chua hotel_review_merged.csv")
print("GEN_MODEL_DIR   :", GEN_MODEL_DIR or "KHONG TIM THAY - can Add Input hotel-mt5-asqp.zip")
print("EXTRACTIVE_CKPT :", EXTRACTIVE_CKPT or "KHONG TIM THAY - can Add Input dual-teacher-output.zip")

assert UNLABELED_CSV, "Thieu hotel_review_merged.csv duoi /kaggle/input."
assert GEN_MODEL_DIR, "Thieu hotel-mt5-asqp (thu muc co model.safetensors truc tiep) duoi /kaggle/input (Add Input hotel-mt5-asqp.zip tu notebook truoc)."
assert EXTRACTIVE_CKPT, "Thieu extractive_teacher.pt duoi /kaggle/input (Add Input dual-teacher-output.zip tu notebook truoc)."

import pandas as pd
_cols = pd.read_csv(UNLABELED_CSV, nrows=0).columns.tolist()
print("Cac cot trong UNLABELED_CSV:", _cols)
if "language" not in _cols:
    print("CANH BAO: khong co cot 'language' - moi review se bi coi la tieng Anh, "
          "View 2/3 (dich) se thanh no-op. Kiem tra lai file da qua add_language.py chua.")


In [ ]:
%cd /kaggle/working/MultiLABSA

# 3 model cung resident (hotel-mT5, XLM-R, NLLB-200) + buoc
# compute_transition_scores cua mT5 (stack full-vocab logits qua tung buoc
# decode) rat de OOM tren T4 14.56GiB neu batch_size/num_beams lon - giam ca
# hai xuong (batch=2, beams=2) va bat expandable_segments de bot phan manh.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

# Thu truoc voi mot tap nho de kiem tra pipeline chay dung, roi moi tang dan.
# Dat MAX_SAMPLES = None de chay toan bo corpus (rat lau, xem canh bao o tren).
MAX_SAMPLES = 300

max_samples_arg = f"--max_unlabeled_samples {MAX_SAMPLES}" if MAX_SAMPLES else ""

!python train.py \
    --unlabeled_csv "{UNLABELED_CSV}" \
    --generative_model "{GEN_MODEL_DIR}" \
    --extractive_backbone xlm-roberta-base \
    --skip_training --extractive_checkpoint "{EXTRACTIVE_CKPT}" \
    --multiview --translator_model facebook/nllb-200-distilled-600M \
    --lang_column language \
    --gen_num_beams 2 \
    --inference_batch_size 2 \
    {max_samples_arg} \
    --alpha 0.4 --beta 0.4 --gamma 0.2 --final_score_threshold 0.5 \
    --output_dir /kaggle/working/checkpoints/multiview \
    --pseudo_labels_out /kaggle/working/checkpoints/multiview/pseudo_labels_multiview.json


In [ ]:
import json, os

pseudo_path = "/kaggle/working/checkpoints/multiview/pseudo_labels_multiview.json"
if os.path.exists(pseudo_path):
    with open(pseudo_path, encoding="utf-8") as f:
        pseudo = json.load(f)
    total_quads = sum(len(r["quads"]) for r in pseudo)
    print(f"So review co pseudo-label: {len(pseudo)} | tong so quad giu lai: {total_quads}")
    for item in pseudo[:3]:
        print(json.dumps(item, ensure_ascii=False, indent=2))
else:
    print("Khong co pseudo_labels_multiview.json - kiem tra log cell tren xem train.py co loi khong.")


In [ ]:
import shutil

shutil.make_archive("/kaggle/working/multiview-pseudo-labels", "zip", "/kaggle/working/checkpoints/multiview")
print("Saved /kaggle/working/multiview-pseudo-labels.zip")
